In [267]:
from matplotlib import pyplot as plt
import pandas as pd
import numpy as np
import pygris

import geopandas as gpd

pd.set_option('display.max_rows', 50)
pd.options.mode.chained_assignment = None


In [268]:
tract_cols = [
    'NAME',
    'ALAND',
    'AWATER',
    'geometry',
]

# Downloading 2010 and 2020 tracts for Cuyahoga County, Ohio.
tracts_2010 = pygris.tracts(state="39", county="035", year=2015)[tract_cols]
tracts_2010['NAME'] = tracts_2010['NAME'].astype(str)

tracts_2020 = pygris.tracts(state="39", county="035", year=2020)[tract_cols]
tracts_2020['NAME'] = tracts_2020['NAME'].astype(str)

# Excluding tract 9900, which represents a water geometry over lake erie
tracts_2010 = tracts_2010[tracts_2010['NAME'] != '9900']
tracts_2020 = tracts_2020[tracts_2020['NAME'] != '9900']

In [296]:
tracts_allign_df = pd.DataFrame(columns=['NAME_10','NAME_20','OL_PROP'])

# Function handling the calculation of the overlap area
ol_area_fn = lambda intersect_row: (row.geometry.intersection(intersect_row.geometry).area / row.geometry.area)

for i in range(tracts_2020.shape[0]):
    row = tracts_2020.iloc[i]
    intersect = tracts_2010[tracts_2010.geometry.intersects(row.geometry)]

    intersect['NAME_20'] = row['NAME']
    intersect['OL_PROP'] = intersect.apply(ol_area_fn, axis=1)
    intersect = intersect.drop(columns=['geometry','ALAND','AWATER']).rename(columns={'NAME': 'NAME_10'})
    intersect = intersect[intersect['OL_PROP'] > 0.08]

    tracts_allign_df = pd.concat([tracts_allign_df, intersect])

# Adding a found special case to the dataframe
tracts_allign_df.loc[len(tracts_allign_df)] = ['1948', '1971', 1.0]

C:\Users\ethan\AppData\Local\Temp\ipykernel_12508\2191682869.py:16: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  tracts_allign_df = pd.concat([tracts_allign_df, intersect])


In [328]:
tracts_allign_df

,NAME_10,NAME_20,OL_PROP
1496,1959,1959,0.993546
1500,1093.01,1093.01,1.000000
1498,1098.01,1098.01,1.000000
1499,1962,1962,0.999984
1497,1963,1963,0.999439
...,...,...,...
1310,1275.01,1275.01,0.997198
1696,1121,1121,1.000000
1695,1235.02,1235.02,1.000000
1698,1024.02,1024.02,0.953016


In [327]:
tracts_allign_df['NAME_10'].value_counts().head(50)

NAME_10
1172.02    2
1311.02    2
1071.01    2
1012       2
1751.03    2
1361.02    2
1606.01    2
1751.04    2
1905.04    2
1721.03    2
1234       1
1891.08    1
1222       1
1891.10    1
1221       1
1891.07    1
1891.05    1
1881.06    1
1959       1
1905.02    1
1881.03    1
1871.06    1
1871.05    1
1219       1
1218       1
1217       1
1215       1
1212       1
1211       1
1208.01    1
1204       1
1202       1
1213       1
1891.11    1
1235.01    1
1905.03    1
1191       1
1417       1
1401       1
1712.06    1
1206       1
1063       1
1064       1
1108.01    1
1105.01    1
1187       1
1049       1
1231       1
1046       1
1043       1
Name: count, dtype: int64

In [297]:
# Get number of unique 2010 and 2020 tracts
num_2010 = len(tracts_allign_df['NAME_10'].unique())
num_2020 = len(tracts_allign_df['NAME_20'].unique())

print(num_2010,num_2020)
print(tracts_2010['NAME'].unique().shape[0],tracts_2020['NAME'].unique().shape[0])

445 426
446 427


In [263]:
t_dict = {}
for index, row in same_tracts_df.iterrows():
    t_dict[row['NAME_10']] = row['NAME_20']
for index, row in diff_tracts_df.iterrows():
    t_dict[row['NAME_10']] = row['NAME_20']

In [264]:
tracts_2010_new = tracts_2010.copy()
tracts_2010_new['NAME'] = tracts_2010_new['NAME'].map(t_dict)